# jepa-as-a-tool: evaluation with world model v2

Muse Spark plays PushT tasks in 3 settings: alone, with the world model, with the real physics.
3 families × 50 tasks × 3 settings = 450 episodes, ~2–3 h, ~$5–9 in contributor mode.

**Before running:**
1. **No GPU needed.** Kaggle: Accelerator *None*, Internet *on*. Colab: any runtime.
2. **The key, as a secret** (never in a cell):
   - Kaggle: Add-ons → Secrets → `META_API_KEY`, ticked for this notebook.
   - Colab: the 🔑 panel → `META_API_KEY`, with notebook access on.
3. **The trained model**: `wm-v2.zip`.
   - Kaggle: *Add Input* → *Upload* → a new dataset with `wm-v2.zip` (Kaggle may unzip it; both work).
   - Colab: drop `wm-v2.zip` into `/content` from the Files panel.

**Long run tip (Kaggle):** *Save Version → Save & Run All* keeps running with the browser closed.

In [ ]:
import os

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    WORK = os.getcwd()
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .

pip's complaints about other packages (gradio, bigframes, ...) are harmless. If it says it upgraded **numpy** or **torch**, restart the kernel and run the cell above again.

## 1. API key and model choice

In [ ]:
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except ImportError:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name)
    except ImportError:
        pass
    from getpass import getpass

    return getpass(f"{name}: ")


os.environ["META_API_KEY"] = secret("META_API_KEY")
os.environ["META_API_BASE_URL"] = "https://api.meta.ai/v1"
# Contributor mode: same model, ~10x cheaper; Meta may train on these requests.
os.environ["META_MODEL"] = "muse-spark-1.3-contributor"
print("key loaded:", bool(os.environ["META_API_KEY"]), "| model:", os.environ["META_MODEL"])

## 2. Find the trained model

Looks for `wm-v2.zip` (or its unzipped folder) and puts it in `runs/wm-v2`.

In [ ]:
import glob
import shutil
import zipfile

TARGET = "runs/wm-v2"
if not os.path.isfile(f"{TARGET}/predictor.pt"):
    roots = ["/kaggle/input", WORK]
    zips = [p for r in roots for p in glob.glob(f"{r}/**/wm-v2*.zip", recursive=True)]
    folders = [
        os.path.dirname(p)
        for r in roots
        for p in glob.glob(f"{r}/**/predictor.pt", recursive=True)
        if os.path.isfile(os.path.join(os.path.dirname(p), "translator.pt"))
    ]
    if zips:
        zipfile.ZipFile(zips[0]).extractall("runs")
    elif folders:
        shutil.copytree(folders[0], TARGET)
    else:
        raise FileNotFoundError("wm-v2.zip not found: see 'Before running', step 3.")
!ls -lh {TARGET}

## 3. Smoke test: 1 episode (~1–2 min, a few cents)

Checks the key, the model and the world model before the long run.
It should end with a one-line table.

In [ ]:
!jat eval --out results/smoke --families rotate --n 1 --settings wm_tool --wm runs/wm-v2

## 4. Evaluation (~2–3 h)

One line per finished episode. If it stops (network, session), run the cell again:
finished episodes are skipped.

In [ ]:
!jat eval --out results/eval-wm-v2 --families translate,rotate,pose --n 50 \
    --settings no_tool,wm_tool,oracle_tool --wm runs/wm-v2 --workers 8

## 5. Send back

Download `eval-wm-v2.zip` and send it, with the table printed above.

In [ ]:
!jat report results/eval-wm-v2
!cd results && zip -q -r {WORK}/eval-wm-v2.zip eval-wm-v2
!ls -lh {WORK}/eval-wm-v2.zip